In [ ]:
PRECOMPILED_EMBEDDINGS_PATH = "qd/datasets/track_embeddings_metrics_32dim_rngMixDS_tita_1.npz"
EMBEDDING_MODEL_PATH = "qd/pretrained_models/model_metrics_VAE/model_metrics_VAE_mixRng_tita_1.pth"

In [ ]:
# Generate and evaluate 10 random tracks
import requests
import random

randomizer = random.Random(100)
num_tracks = 10
seed_candidates = randomizer.sample(range(1, 10000), num_tracks)

def generate_track(seed, randomizer):
    response = requests.post(
        "http://localhost:4242/generate",
        json={
            "id": seed,
            "mode": "voronoi",
            "trackSize": randomizer.randint(4, 10),
            "rngMode": 0,
        },
        timeout=60,
    )

    if not response.ok:
        raise Exception(f"API error {response.status_code}: {response.text}")

    sol = response.json()
    sol["rngMode"] = 1
    return sol

def eval_track(sol):
    response = requests.post(
        "http://localhost:4242/evaluate",
        json=sol,
        timeout=60,
    )
    if not response.ok:
        raise Exception(f"API error {response.status_code}: {response.text}")

    r_json = response.json()
    return r_json.get("fitness", {})

tracks_results = []
for seed in seed_candidates:
    try:
        sol = generate_track(seed, randomizer)
        fit = eval_track(sol)
        embedding_data = fit.get("embedding_data", [])

        if embedding_data is None or len(embedding_data) == 0:
            print(f"Seed {seed}: missing embedding_data, skipped")
            continue

        tracks_results.append({
            "seed": seed,
            "solution": sol,
            "fitness": fit,
            "embedding_data": embedding_data,
        })
    except Exception as e:
        print(f"Seed {seed}: error during generation/evaluation - {e}")

print(f"Generated and evaluated {len(tracks_results)} tracks out of {num_tracks} requested")
print("Seeds used:", [item["seed"] for item in tracks_results])

In [ ]:
# cleaned_metrics = tracks_results

print("cleaned_metrics shape:", cleaned_metrics.shape)

if cleaned_metrics.ndim != 3 or cleaned_metrics.shape[2] != 3:
    raise ValueError(
        f"Expected cleaned_metrics with shape (num_signals, timesteps, 3), got {cleaned_metrics.shape}"
    )

dimension_names = ["speed", "steering", "distance_to_border"]

# For each metric, show magnitude and phase side by side
for dim_idx, dim_name in enumerate(dimension_names):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharex=True)
    ax_mag, ax_phase = axes

    for signal_idx, metrics in enumerate(cleaned_metrics):
        signal = metrics[:, dim_idx]
        spectrum = np.fft.rfft(signal)

        fft_mag = np.abs(spectrum)
        fft_phase = np.unwrap(np.angle(spectrum))

        label = f"Roll {int(roll_percents[signal_idx] * 100)}%"
        ax_mag.plot(fft_mag, label=label)
        ax_phase.plot(fft_phase, label=label)

    ax_mag.set_title(f"FFT Magnitude - {dim_name}")
    ax_mag.set_xlabel("Frequency Bin")
    ax_mag.set_ylabel("Magnitude")
    ax_mag.grid(True)
    ax_mag.legend()

    ax_phase.set_title(f"FFT Phase - {dim_name}")
    ax_phase.set_xlabel("Frequency Bin")
    ax_phase.set_ylabel("Phase (radians)")
    ax_phase.grid(True)
    ax_phase.legend()

    fig.tight_layout()
    plt.show()